# Module 2 — Class 4: Pandas for Data Analysis

**Lecture:** [https://bepro-aiml.github.io/aiml-platform/#/module/2/class/4](https://bepro-aiml.github.io/aiml-platform/#/module/2/class/4)

## How to use this notebook

Every section has the same shape:
1. **Concept** — short explanation from the lecture.
2. **Predict** — write what you think will happen *before* running the code.
3. **Run** — execute the cell.
4. **Explain** — write 1–2 sentences in your own words about what the code did and why.
5. **Practice** — your turn. The grade is on the markdown cells, not just running the code.

Skipping the markdown cells = automatic 0 on the in-class checkpoint. Run code without thinking and you will be lost by Class 5.

---
## 1. Series and DataFrames

**Concept.** Pandas is a programmable spreadsheet. `Series` = one labeled column. `DataFrame` = 2D labeled table with an index. 90% of ML work starts with loading, cleaning, and engineering features in a DataFrame.

### 🤔 Predict

Before running the next two cells, answer in this cell:
1. What do you expect the output of `df` to look like? Sketch it in markdown (use a table).
2. What is the *index* of `df` going to be — and what does that even mean?

**Your prediction:**

In [1]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Ali", "Zara", "Jasur"],
    "age":  [22, 28, 31],
    "city": ["Tashkent", "Samarkand", "Bukhara"]
})
df

,name,age,city
0,Ali,22,Tashkent
1,Zara,28,Samarkand
2,Jasur,31,Bukhara


In [2]:
print(type(df['age']))
print(type(df[['name', 'age']]))
print(df.index)

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex(start=0, stop=3, step=1)


### ✍️ Explain

Answer all three in your own words (one sentence each):
1. Why is `df['age']` a Series but `df[['name', 'age']]` a DataFrame?
2. What is the default index when you don't set one?
3. When would you *want* a non-default index? Give one realistic example.

**Your answer:**

1. df['age'] is a Series because it selects a single column, while df[['name', 'age']] is a DataFrame because it selects multiple columns.
2. The default index is a sequence of integers starting from 0.
3. You would want a non-default index when a column has unique identifiers, for example using user IDs as the index in a dataset of users.

### 🛠 Practice

Build a DataFrame with at least 5 rows and 4 columns about something you actually care about (your music playlist, your subjects this semester, anything). Then:
- Set one of the columns as the index using `df.set_index('col_name')`.
- Show what `df.loc[<one of your index labels>]` returns.
- In a markdown cell, explain *why* `df.loc` finds that row even though it's not row 0.

In [3]:
df = pd.DataFrame({
    'subject': ['Math', 'Physics', 'English', 'History', 'Biology'],
    'grade': [90, 85, 88, 92, 80],
    'teacher': ['Mr A', 'Mr B', 'Ms C', 'Ms D', 'Mr E'],
    'hours': [5, 4, 3, 2, 4]
})

df = df.set_index('subject')

df.loc['Math']

# df.loc finds rows using the index label, not the position, so it returns the row for 'Math' even though it is not row 0.

,Math
grade,90
teacher,Mr A
hours,5


---
## 2. Loading and Exploring

**Concept.** Five inspection commands you will run on every dataset for the rest of your career: `head`, `shape`, `info`, `describe`, `isna().sum()`. Each one answers a different question.

### 🤔 Predict

For each of the five commands below, write **one sentence** about what *question* it answers — before running anything.

1. `df.head()` answers: It shows the first few rows of the dataset to preview the data.

2. `df.shape` answers: It shows the number of rows and columns in the dataset.

3. `df.info()` answers: It provides information about the dataset, including column names, data types, and non-null values.

4. `df.describe()` answers: It shows summary statistics of numerical columns, such as mean, min, max, and quartiles.

5. `df.isna().sum()` answers: It shows the number of missing (null) values in each column.

In [22]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame

# Show the first 5 rows of the dataset (quick preview of the data)
df.head()

# Display dataset info: column names, data types, and missing values
df.info()

# Generate descriptive statistics (mean, min, max, etc. for numeric columns)
df.describe()

# Show the shape of the dataset (number of rows and columns)
df.shape

# List all column names in the dataset
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target'],
      dtype='object')

I compared my predictions with the actual outputs of each command.

Most of my predictions were correct. However, I was a bit surprised by the output of df.describe(), because it showed only statistics for numeric columns, not all columns, which I did not fully expect at first.

### ✍️ Explain

Compare your predictions above to what each command actually printed. For any prediction that was wrong, write what surprised you.

If a date column is stored as a string, pandas treats it like text. After using parse_dates, it becomes a datetime, which allows correct sorting and analysis. For example, string sorting can be wrong, but datetime sorting is chronological.

### 🛠 Practice

If your CSV has a column that *looks* like a date but `df.dtypes` shows it as `object`:
- Re-load with `pd.read_csv('...', parse_dates=['that_column'])`.
- Run `df.dtypes` again.
- In a markdown cell, explain why this matters. (Hint: try sorting by date when it's a string vs when it's a datetime.)

---
## 3. Filtering, GroupBy, Merging

These are the three operations every data role uses every day. Master them and the rest of Pandas falls into place.

### 3a. Selection — `loc` vs `iloc`

**Concept.** `loc` selects by **label**. `iloc` selects by **integer position**. They behave the same only when your index happens to be `0, 1, 2, …` — once you set a meaningful index, they diverge.

### 🤔 Predict

Given the `df` from Section 1 (Ali, Zara, Jasur):
- What does `df.loc[0]` return?
- What does `df.iloc[0]` return?
- Now imagine I run `df = df.set_index('name')`. After that:
  - What does `df.loc['Ali']` return?
  - What does `df.iloc[0]` return?
  - What does `df.loc[0]` return?  *(careful — think before answering)*

In [5]:
df = pd.DataFrame({
    "name": ["Ali", "Zara", "Jasur"],
    "age":  [22, 28, 31],
    "city": ["Tashkent", "Samarkand", "Bukhara"]
})

print('--- before set_index ---')
print(df.loc[0])
print(df.iloc[0])

df = df.set_index('name')
print('--- after set_index ---')
print(df.loc['Ali'])
print(df.iloc[0])
# Try uncommenting the next line and explain the error in markdown below:
# print(df.loc[0])

--- before set_index ---
name         Ali
age           22
city    Tashkent
Name: 0, dtype: object
name         Ali
age           22
city    Tashkent
Name: 0, dtype: object
--- after set_index ---
age           22
city    Tashkent
Name: Ali, dtype: object
age           22
city    Tashkent
Name: Ali, dtype: object


### ✍️ Explain

1. Which of your predictions were wrong? Why?
2. The commented `df.loc[0]` would raise an error. What error, and why does it make sense?
3. *Quick Check from the lecture:* In one sentence — when do you reach for `loc` and when for `iloc`?

### 3b. Boolean Filtering

### 🤔 Predict

Three of these four lines are wrong. Without running them, decide which one is correct and explain *why* the others fail.

```python
# A
df[df.age >= 18 and df.city == 'Tashkent']
# B
df[(df.age >= 18) & (df.city == 'Tashkent')]
# C
df[df.age >= 18 && df.city == 'Tashkent']
# D
df.filter(age=18, city='Tashkent')
```

**Correct one:** B

**Why each wrong one fails:**

A - fails because 'and' cannot be used with pandas Series.

C - fails because '&&' is not valid syntax in Python.

D - fails because df.filter is not used for row filtering with conditions.

In [6]:
# Reload df without the index modification from earlier
df = pd.DataFrame({
    "name": ["Ali", "Zara", "Jasur", "Madina", "Otabek"],
    "age":  [22, 28, 31, 17, 45],
    "city": ["Tashkent", "Samarkand", "Bukhara", "Tashkent", "Samarkand"]
})

# Test the four candidates one at a time. Comment out 3, uncomment 1, run, repeat.

# df[df.age >= 18 and df.city == 'Tashkent']
# df[(df.age >= 18) & (df.city == 'Tashkent')]
# df[df.age >= 18 && df.city == 'Tashkent']
# df.filter(age=18, city='Tashkent')

### 🛠 Practice

Write **three** boolean filters on `df`:
1. People over 25 *and* not in Tashkent.
2. People in Tashkent *or* in Bukhara.
3. The youngest person (use a comparison against `df.age.min()`).

After each filter, write one sentence about *who* the filter selected and *why*.

In [7]:
# Filter 1
df[(df['age'] > 25) & (df['city'] != 'Tashkent')]
# This filter selects people older than 25 who are not in Tashkent because both conditions must be true.

# Filter 2
df[(df['city'] == 'Tashkent') | (df['city'] == 'Bukhara')]
# This filter selects people who are in Tashkent or Bukhara because at least one condition must be true.

# Filter 3
df[df['age'] == df['age'].min()]
# This filter selects the youngest person because it compares ages to the minimum value.


,name,age,city
3,Madina,17,Tashkent


### 3c. GroupBy — split-apply-combine

In [8]:
df.groupby('city')['age'].mean()

,age
city,
Bukhara,31.0
Samarkand,36.5
Tashkent,19.5


### ✍️ Explain — *the lecture's Quick Check #3*

What type does the cell above return? Choose one and justify in 2 sentences:
- DataFrame
- Series indexed by city
- list
- scalar

Now run `type(df.groupby('city')['age'].mean())` to verify.

In [9]:
# Run this to compare
result = df.groupby('city')['age'].mean()
print(type(result))
print(result)

<class 'pandas.core.series.Series'>
city
Bukhara      31.0
Samarkand    36.5
Tashkent     19.5
Name: age, dtype: float64


### 🛠 Practice

Use `.agg()` with a dict to compute, in **one** call:
- average `age` per city
- count of names per city
- the *youngest* person's age per city (use `'min'`)

In [10]:
# Your single-call .agg() here


### 3d. Merging

### 🤔 Predict

I have two DataFrames. Customers (5 rows) and Orders (3 rows, only some customers have ordered). I run:

```python
pd.merge(customers, orders, on='customer_id', how='left')
```

Predict:
- How many rows in the result?
- For customers without orders, what fills the order columns?
- If I changed `how='inner'`, how many rows would I get?

In [11]:
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Ali', 'Zara', 'Jasur', 'Madina', 'Otabek']
})
orders = pd.DataFrame({
    'customer_id': [1, 1, 3],
    'amount': [50000, 75000, 30000]
})

left = pd.merge(customers, orders, on='customer_id', how='left')
inner = pd.merge(customers, orders, on='customer_id', how='inner')

print('--- LEFT join ---')
print(left)
print()
print('--- INNER join ---')
print(inner)

--- LEFT join ---
   customer_id    name   amount
0            1     Ali  50000.0
1            1     Ali  75000.0
2            2    Zara      NaN
3            3   Jasur  30000.0
4            4  Madina      NaN
5            5  Otabek      NaN

--- INNER join ---
   customer_id   name  amount
0            1    Ali   50000
1            1    Ali   75000
2            3  Jasur   30000


### ✍️ Explain

1. Why does the LEFT join have 6 rows but the INNER has only 3?
2. Notice that customer 1 appears twice in LEFT — explain why.
3. *Quick Check from the lecture:* In one sentence, what is the rule for what happens to unmatched left rows in a LEFT join?

### 🛠 Practice — capstone for today

Combine everything in this section:
1. Add a `region` lookup DataFrame (Tashkent → 'Capital', Samarkand → 'South', Bukhara → 'South', etc.).
2. Merge it into your customer DataFrame.
3. GroupBy region and compute total order `amount` per region (combine with the orders DataFrame).
4. Filter to show only regions with total > 80000.
5. In a markdown cell, write 2–3 sentences explaining the business interpretation of your result.

In [12]:
customers = pd.DataFrame({
    'name': ['Ali', 'Zara', 'Jasur', 'Madina', 'Otabek'],
    'city': ['Tashkent', 'Samarkand', 'Bukhara', 'Tashkent', 'Samarkand']
})

orders = pd.DataFrame({
    'name': ['Ali', 'Ali', 'Jasur'],
    'amount': [50000, 75000, 30000]
})

regions = pd.DataFrame({
    'city': ['Tashkent', 'Samarkand', 'Bukhara'],
    'region': ['Capital', 'South', 'South']
})

df_full = customers.merge(regions, on='city').merge(orders, on='name', how='left')

result = df_full.groupby('region')['amount'].sum()

result[result > 80000]

,amount
region,
Capital,125000.0


---
## End-of-class checkpoint quiz

Answer all 5 in markdown cells. These are the same 5 questions from the lecture page — bring written answers to the next class.

1. `loc` vs `iloc` — which is which, and when do you reach for each?
2. Why does `df[df.age >= 18 and df.city == 'T']` raise an error, and how do you fix it?
3. What does `df.groupby('city')['age'].mean()` return — DataFrame, Series, list, or scalar?
4. In a left join, what happens to rows in the *left* table that have no match in the right table?
5. A 5GB CSV crashes your laptop. What is the first move? (Two valid answers — give both.)

## Homework from pdf

# Task 1.1


In [13]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Ali", "Zara", "Jasur"],
    "age": [22, 28, 31],
    "city": ["Tashkent", "Samarkand", "Bukhara"]
})

print(type(df["age"]))
print(type(df[["name", "age"]]))
print(df.index)

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex(start=0, stop=3, step=1)


# Task 1.2


In [14]:
new_rows = pd.DataFrame({
    "name": ["Madina", "Otabek", "Sardor"],
    "age": [24, 35, 27],
    "city": ["Khiva", "Namangan", "Andijan"]
})

df = pd.concat([df, new_rows], ignore_index=True)
df

,name,age,city
0,Ali,22,Tashkent
1,Zara,28,Samarkand
2,Jasur,31,Bukhara
3,Madina,24,Khiva
4,Otabek,35,Namangan
5,Sardor,27,Andijan


# Task 2.1


In [15]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [16]:



# Task 2.2
df.head()
df.shape
df.info()
df.describe()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


,0
sepal length (cm),0
sepal width (cm),0
petal length (cm),0
petal width (cm),0
target,0


In [17]:
# Task 3.1
df["sepal length (cm)"]
df[["sepal length (cm)", "sepal width (cm)"]]
df.loc[0]
df.iloc[0]

,0
sepal length (cm),5.1
sepal width (cm),3.5
petal length (cm),1.4
petal width (cm),0.2
target,0.0


In [18]:
# Task 3.2
df[(df["sepal length (cm)"] > 5) & (df["target"] == 0)]
df[(df["sepal width (cm)"] > 3) | (df["target"] == 2)]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
5,5.4,3.9,1.7,0.4,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [19]:
# Task 3.3
df.groupby("target")["sepal length (cm)"].mean()
df.groupby("target").agg({
    "sepal length (cm)": "mean",
    "sepal width (cm)": "count"
})

,sepal length (cm),sepal width (cm)
target,,
0,5.006,50
1,5.936,50
2,6.588,50


In [20]:
# Task 3.4
lookup = pd.DataFrame({
    "target": [0, 1, 2],
    "category": ["Setosa", "Versicolor", "Virginica"]
})

merged = pd.merge(df, lookup, on="target", how="left")
merged

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,category
0,5.1,3.5,1.4,0.2,0,Setosa
1,4.9,3.0,1.4,0.2,0,Setosa
2,4.7,3.2,1.3,0.2,0,Setosa
3,4.6,3.1,1.5,0.2,0,Setosa
4,5.0,3.6,1.4,0.2,0,Setosa
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2,Virginica
146,6.3,2.5,5.0,1.9,2,Virginica
147,6.5,3.0,5.2,2.0,2,Virginica
148,6.2,3.4,5.4,2.3,2,Virginica
